In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import re

## Define nodes
Filter for Downtown Line Stations

In [2]:
nodes = pd.read_csv('mrt_lrt_stations.csv')
# only include Down Town Line stations in the list.
dtl_stations = [station for station in nodes['ALPHANUMERIC_CODE'] if station.startswith('DT')]
print(len(dtl_stations))
dtl_stations

35


['DT1',
 'DT2',
 'DT3',
 'DT4',
 'DT5',
 'DT6',
 'DT7',
 'DT8',
 'DT9',
 'DT10',
 'DT11',
 'DT12',
 'DT13',
 'DT14',
 'DT15',
 'DT16',
 'DT17',
 'DT18',
 'DT19',
 'DT20',
 'DT21',
 'DT22',
 'DT23',
 'DT24',
 'DT25',
 'DT26',
 'DT27',
 'DT28',
 'DT29',
 'DT30',
 'DT31',
 'DT32',
 'DT33',
 'DT34',
 'DT35']

### Classify DTL Stations by Demand Tier

In [3]:
# 1. Filter for Weekday, 7am-8am Window and only DT stations
df = pd.read_csv("transport_node_train_202603.csv")
dtl_data = df[
    (df['DAY_TYPE'] == 'WEEKDAY') & 
    (df['TIME_PER_HOUR'] == 7) & 
    (df['PT_CODE'].str.contains('DT'))
].copy()

# 2. Standardize PT_CODE to just the 'DTxx' part
# This turns "NS21 / DT11" into "DT11"
dtl_data['PT_CODE'] = dtl_data['PT_CODE'].str.extract(r'(DT\d+)')

# 3. Calculate Throughput
dtl_data['volume'] = dtl_data['TOTAL_TAP_IN_VOLUME'] + dtl_data['TOTAL_TAP_OUT_VOLUME']

# 4. Sort Numerically (Extract number from 'DT1', 'DT2', etc.)
dtl_data['stn_num'] = dtl_data['PT_CODE'].str.extract(r'(\d+)').astype(int)
dtl_data = dtl_data.sort_values('stn_num')

# 5. Create the sorted volume dictionary
sorted_volumes = dict(zip(dtl_data['PT_CODE'], dtl_data['volume']))

# 6. Divide into Tiers based on volume (High, Mid, Low)
high_thresh = dtl_data['volume'].quantile(0.8)  # Top 20%
# threshold can be changed later. i used a 20% cut-off to tighten tier 1 
# because if tier 1 has too many stations it would be force to stop at too many stations :(
low_thresh = dtl_data['volume'].quantile(0.3)   # Bottom 30%

tier_high = set(dtl_data[dtl_data['volume'] >= high_thresh]['PT_CODE'])
tier_mid = set(dtl_data[(dtl_data['volume'] < high_thresh) & (dtl_data['volume'] >= low_thresh)]['PT_CODE'])
tier_low = set(dtl_data[dtl_data['volume'] < low_thresh]['PT_CODE'])

# Step 7: Manual addition
# These must be Tier 1 to preserve network connectivity
interchanges_dtl = {
    'DT1', 'DT9', 'DT10', 'DT11', 'DT12', 'DT14', 
    'DT15', 'DT16', 'DT19', 'DT26', 'DT32', 'DT35'
}
# Force them into Tier 1
tier_high.update(interchanges_dtl)

# Remove them from other tiers to ensure no duplicates
tier_mid.difference_update(interchanges_dtl)
tier_low.difference_update(interchanges_dtl)

print("Station Volumes:", sorted_volumes)
print("Tier 1:", tier_high)
print("Tier 2:", tier_mid)
print("Tier 3:", tier_low)
print("Total Stations:", len(tier_high) + len(tier_mid) + len(tier_low))

Station Volumes: {'DT1': 216813, 'DT2': 25996, 'DT3': 32897, 'DT4': 11434, 'DT5': 58113, 'DT6': 32699, 'DT7': 23770, 'DT8': 43803, 'DT9': 21539, 'DT10': 23752, 'DT11': 167655, 'DT12': 67344, 'DT13': 19718, 'DT14': 75772, 'DT15': 25717, 'DT16': 41395, 'DT17': 44111, 'DT18': 32717, 'DT19': 57668, 'DT20': 19712, 'DT21': 17161, 'DT22': 28894, 'DT23': 16493, 'DT24': 24057, 'DT25': 20786, 'DT26': 63868, 'DT27': 46914, 'DT28': 64299, 'DT29': 39386, 'DT30': 45449, 'DT31': 44461, 'DT32': 195947, 'DT33': 106807, 'DT34': 24156, 'DT35': 38070}
Tier 1: {'DT33', 'DT32', 'DT16', 'DT15', 'DT9', 'DT14', 'DT28', 'DT1', 'DT19', 'DT10', 'DT35', 'DT26', 'DT12', 'DT11'}
Tier 2: {'DT31', 'DT18', 'DT22', 'DT30', 'DT6', 'DT17', 'DT5', 'DT27', 'DT2', 'DT29', 'DT8', 'DT3'}
Tier 3: {'DT24', 'DT23', 'DT4', 'DT13', 'DT34', 'DT20', 'DT25', 'DT7', 'DT21'}
Total Stations: 35


In [4]:
dtl_data

,YEAR_MONTH,DAY_TYPE,TIME_PER_HOUR,PT_TYPE,PT_CODE,TOTAL_TAP_IN_VOLUME,TOTAL_TAP_OUT_VOLUME,volume,stn_num
5845,2026-03,WEEKDAY,7,TRAIN,DT1,147690,69123,216813,1
2784,2026-03,WEEKDAY,7,TRAIN,DT2,7580,18416,25996,2
115,2026-03,WEEKDAY,7,TRAIN,DT3,22419,10478,32897,3
5081,2026-03,WEEKDAY,7,TRAIN,DT4,8271,3163,11434,4
2416,2026-03,WEEKDAY,7,TRAIN,DT5,40268,17845,58113,5
3832,2026-03,WEEKDAY,7,TRAIN,DT6,12378,20321,32699,6
4488,2026-03,WEEKDAY,7,TRAIN,DT7,7566,16204,23770,7
1649,2026-03,WEEKDAY,7,TRAIN,DT8,9307,34496,43803,8
4000,2026-03,WEEKDAY,7,TRAIN,DT9,9667,11872,21539,9
4176,2026-03,WEEKDAY,7,TRAIN,DT10,9369,14383,23752,10


## Define qj

In [5]:
TRAIN_COUNT = 24 # Total trains in an hour
m = gp.Model('minimize total passenger travel time + waiting penalty')
STATION_COUNT = len(dtl_stations)
# create a TRAIN_COUNT by STATION_COUNT matrix of binary decision variables, where x[t][j] = 1 if train t stops at station j, and 0 otherwise
x = [[0 for j in range(STATION_COUNT)] for t in range(TRAIN_COUNT)]
# Create decision variables for each station and each train, where x[t][j] = 1 if train t stops at station j, and 0 otherwise
for t in range(TRAIN_COUNT):
    for j in range(STATION_COUNT):
        x[t][j] = m.addVar(vtype=GRB.BINARY, name=f'x_{t}_{j}')

y = [0 for t in range(TRAIN_COUNT)]       
# y[t] is binary, 1 if train t is express, 0 if train t is regular, local train
for t in range(TRAIN_COUNT):
    y[t] = m.addVar(vtype=GRB.BINARY, name=f'y_{t}')
z = [0 for j in range(STATION_COUNT)]
# z[j] is binary, 1 if station j is designated as express station, 0 otherwise
for j in range(STATION_COUNT):
    z[j] = m.addVar(vtype=GRB.BINARY, name=f'z_{j}')

u = [0 for j in range(STATION_COUNT)]
# u[j] >= 0 is the service shortfall at station j
# if u[j] = 0, station j gets enough stopping trains
# if u[j] > 0, station j is under-served and incurs a penalty
for j in range(STATION_COUNT):
    u[j] = m.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f'u_{j}')

print(len(x[0]))

# q[j] = average onboard passengers affected when a train stops at station j.
# It is defined here so Constraint 6 can use q before the objective section.
df_raw = pd.read_csv('origin_destination_train_202603.csv')

def get_final_q_list(df, dtl_stations, num_weekdays=22, num_trains=TRAIN_COUNT):
    mask = (
        (df['DAY_TYPE'] == 'WEEKDAY') &
        (df['TIME_PER_HOUR'] == 7) &
        (df['ORIGIN_PT_CODE'].str.contains('DT', na=False)) &
        (df['DESTINATION_PT_CODE'].str.contains('DT', na=False))
    )
    df_filtered = df[mask].copy()

    def extract_dt(val):
        match = re.search(r'DT\d+', str(val))
        return match.group(0) if match else None

    df_filtered['ORIGIN_CLEAN'] = df_filtered['ORIGIN_PT_CODE'].apply(extract_dt)
    df_filtered['DEST_CLEAN'] = df_filtered['DESTINATION_PT_CODE'].apply(extract_dt)

    q_list = []

    # for station j, count trips that start before j and end at or after j. these passengers are onboard when the train reaches station j
    for i, current_stat in enumerate(dtl_stations):
        before_stations = dtl_stations[:i]
        at_or_after_stations = dtl_stations[i:]

        onboard_mask = (
            df_filtered['ORIGIN_CLEAN'].isin(before_stations) &
            df_filtered['DEST_CLEAN'].isin(at_or_after_stations)
        )

        total_load = df_filtered.loc[onboard_mask, 'TOTAL_TRIPS'].sum()
        q_avg = (total_load / num_weekdays) / num_trains 
        # divide by number of weekdays to get average daily load, 
        # then divide by number of trains to get average load per train
        q_list.append(round(q_avg, 4))

    return q_list

q = get_final_q_list(df_raw, dtl_stations)
print(f'q length: {len(q)}')
print(q)

Set parameter Username
Set parameter LicenseID to value 2768791
Academic license - for non-commercial use only - expires 2027-01-22
35
q length: 35
[np.float64(0.0), np.float64(153.5436), np.float64(154.3182), np.float64(173.8731), np.float64(182.1326), np.float64(216.8144), np.float64(218.1402), np.float64(213.3087), np.float64(204.4489), np.float64(202.6686), np.float64(204.1174), np.float64(196.7386), np.float64(183.9034), np.float64(174.2254), np.float64(153.6534), np.float64(133.6951), np.float64(117.0663), np.float64(73.2121), np.float64(49.392), np.float64(49.9186), np.float64(49.5928), np.float64(52.3466), np.float64(70.1439), np.float64(72.8314), np.float64(83.1989), np.float64(88.5398), np.float64(96.9962), np.float64(91.4129), np.float64(88.2576), np.float64(97.7765), np.float64(101.9735), np.float64(102.1193), np.float64(100.1648), np.float64(67.5265), np.float64(52.6951)]


### Constraint 1: Mandatory Terminal Stop
terminal stations must always be served and cannot be skipped.

In [6]:
# station DT1 and DT35 must be served by all trains, express or regular, and cannot be skipped.

for t in range(TRAIN_COUNT):
    m.addConstr(x[t][0] == 1)
    m.addConstr(x[t][STATION_COUNT - 1] == 1)

### Constraint 2: Local Train Requirement
If a train is local, it MUST stop at every station.

In [7]:
# add constraint that says if a train is local, it MUST stop at every station, i.e. xtj >= 1 - yt for all t, for all j

for t in range(TRAIN_COUNT):
    for j in range(STATION_COUNT):
        m.addConstr(x[t][j] >= 1 - y[t]) 

### Constraint 3A: Mandatory Interchange Stops
This cell makes DTL interchange stations mandatory stops for all trains. It represents a network-connectivity policy for choosing mandatory stops.

In [8]:
M = set()

interchanges = [('NS1', 'EW24'), ('NS9', 'TE2'), ('CE0', 'CC4'), ('CE0', 'DT16'),
                ('NS17', 'CC15'), ('NS21', 'DT11'), ('NS22', 'TE14'), 
               ('NS24', 'NE6'), ('NS24', 'CC1'), ('NS25', 'EW13'), 
               ('NS26', 'EW14'), ('NS27', 'TE20'), ('NS27', 'CE2'),
               ('CC22', 'EW21'), ('EW16', 'NE3'), ('EW16', 'TE17'), 
               ('EW12', 'DT14'), ('EW8', 'CC9'), ('EW4', 'CG0'), 
               ('EW2', 'DT32'), ('CG1', 'DT35'), ('CC19', 'DT9'),
               ('DT10', 'TE11'), ('NE7', 'DT12'), ('DT15', 'CC4'),
               ('DT16', 'CE1'), ('DT19', 'NE4'), ('DT26', 'CC10'),
               ('CC29', 'NE1'), ('CC17', 'TE9'), ('CC13', 'NE12'),
               ('CC1', 'NE6'), ('CE2', 'TE20'), ('NE3', 'TE17'), ('DT1', 'BP6')
               ]

for interchange in interchanges:
    if interchange[0] in dtl_stations:
        M.add(interchange[0])
    elif interchange[1] in dtl_stations:
        M.add(interchange[1])

station_to_idx = {f"DT{i}": i - 1 for i in range(1, STATION_COUNT + 1)}
for t in range(TRAIN_COUNT):
    for station in M:
        m.addConstr(x[t][station_to_idx[station]] == 1, name=f"mandatory_stop_{t}_{station}")

print(M)

{'DT32', 'DT26', 'DT16', 'DT1', 'DT19', 'DT15', 'DT10', 'DT14', 'DT35', 'DT9', 'DT12', 'DT11'}


### Constraint 3B: Mandatory Tier 1 Stops
This cell makes all Tier 1 stations mandatory stops for all trains. In the current notebook, this is the main mandatory-stop set used later as `M`; if both Constraint 3A and 3B are run, the resulting mandatory-stop constraints are cumulative.

In [9]:
M = list(tier_high) 
station_to_idx = {code: i for i, code in enumerate(dtl_stations)}

for t in range(TRAIN_COUNT):
    for station in M:
        j = station_to_idx[station]
        m.addConstr(
            x[t][j] == 1, 
            name=f"C3_MandatoryStop_Train{t}_{station}"
        )

print(f"Stations in M: {sorted(M)}")

Stations in M: ['DT1', 'DT10', 'DT11', 'DT12', 'DT14', 'DT15', 'DT16', 'DT19', 'DT26', 'DT28', 'DT32', 'DT33', 'DT35', 'DT9']


### Constraint 4: Express Limit
Limit the Number of Express Trains. At most `K` trains in the one-hour period can be designated as express trains.

In [10]:
K = 6 # Explanation in report

m.addConstr(gp.quicksum(y[j] for j in range(TRAIN_COUNT)) <= K)

<gurobi.Constr *Awaiting Model Update*>

### Constraint 5: Station Skip Eligibility
Express trains can only stop at designated express stations, i.e. if train t is express it can ONLY stop where the station is express, else if local train, no restriction

In [11]:
# xtj ≤ zj + (1 – yt) ∀ t ∈ T, j ∈ S
for t in range(TRAIN_COUNT):
    for j in range(STATION_COUNT):
        m.addConstr(x[t][j] <= z[j] + (1 - y[t]))

### Constraint 6: Maximum Skip Threshold
Limit Skipped Stations Within Each Bay. The DTL is split into bays between mandatory stops. For each bay, the model limits how many stations an express train can skip, using `q_j` to calibrate the maximum number of skippable stations in each bay, with lower-q stations implicitly making a bay more permissive to skipping

In [12]:
# ∑(1 - x[t][j]) <= L_b y[t] for each bay b and train t.
# L_b is calculated bay-by-bay using the q[j] values defined earlier.

# Indexes for mandatory stop stations
m_indices = sorted([station_to_idx[s] for s in M])

# Include terminal stations so the full line is split into bays between mandatory stops.
if 0 not in m_indices:
    m_indices.insert(0, 0)
if (STATION_COUNT - 1) not in m_indices:
    m_indices.append(STATION_COUNT - 1)

m_indices = sorted(set(m_indices))

# Section the line into bays. Each bay contains stations between two mandatory stops.
bays_indices = []
for i in range(len(m_indices) - 1):
    start = m_indices[i] + 1
    end = m_indices[i + 1]
    if start < end:
        bays_indices.append(list(range(start, end)))

# Add up stations that can be skipped in order of increasing q[j]
# until the exclusion threshold for that bay is reached.
exlvl = 0.5  # change to 0.2 or 0.4 depending on the assumption
bay_skip_limits = {}

for b_idx, bay in enumerate(bays_indices):
    bay_q_values = sorted(q[j] for j in bay)
    total_bay_q = sum(bay_q_values)
    current_sum = 0
    lb_limit = 0

    for val in bay_q_values:
        if total_bay_q > 0 and current_sum + val <= exlvl * total_bay_q:
            current_sum += val
            lb_limit += 1
        else:
            break

    bay_skip_limits[b_idx] = lb_limit

    for t in range(TRAIN_COUNT):
        m.addConstr(
            gp.quicksum(1 - x[t][j] for j in bay) <= lb_limit * y[t],
            name=f"bay_limit_{b_idx}_train_{t}"
        )

print('Bay skip limits:', bay_skip_limits)

Bay skip limits: {0: 3, 1: 0, 2: 1, 3: 3, 4: 0, 5: 1, 6: 0}


In [13]:
m_indices

[0, 8, 9, 10, 11, 13, 14, 15, 18, 25, 27, 31, 32, 34]

In [14]:
bays_indices

[[1, 2, 3, 4, 5, 6, 7],
 [12],
 [16, 17],
 [19, 20, 21, 22, 23, 24],
 [26],
 [28, 29, 30],
 [33]]

### Constraint 7: Service Demand Satisfaction
In other words, Service Coverage Target. `H_j` is the target number of trains that should stop at station `j`. If fewer than `H_j` trains stop there, the shortfall is captured by `u_j` and penalized in the objective.

In [15]:
H_list = []

for stn in dtl_data['PT_CODE']:
    if stn in tier_high:
        H_list.append(TRAIN_COUNT)         # 100% Target (24 trains) # TARGETS can be adjusted later, but tier 1 should have 100%
    elif stn in tier_mid:
        H_list.append(int(TRAIN_COUNT * 0.75)) # 75% Target (18 trains)
    else:
        H_list.append(int(TRAIN_COUNT * 0.50)) # 50% Target (12 trains)

H = H_list

for j in range(STATION_COUNT):
    m.addConstr(gp.quicksum(x[t][j] for t in range(TRAIN_COUNT)) + u[j] >= H[j]) 
    # if the number of trains stopping at station j is less than the ideal number H[j], 
    # then uj will be positive and will pay a penalty in the objective function. if the number of trains stopping at station j is greater than or equal to H[j], then uj will be 0 and there will be no penalty in the objective function.

print("Hj for each station j:", H)


Hj for each station j: [24, 18, 18, 12, 18, 18, 12, 18, 24, 24, 24, 24, 12, 24, 24, 24, 18, 18, 24, 12, 12, 18, 12, 12, 12, 24, 18, 24, 18, 18, 18, 24, 24, 12, 24]


### Constraint 8: Common Express Stopping Pattern
All express trains must follow the same skip-stop route. E.g. if we have ABCDE as train stations, and the express station are ACE, then all express trains have to follow ACE.

In [16]:
# All express trains must follow the same skip-stop route

for t in range(TRAIN_COUNT):
    for j in range(STATION_COUNT):
        # If train t is express, it must stop if station j is an express station
        m.addConstr(x[t][j] >= z[j] - (1 - y[t]), name=f"express_lower_{t}_{j}")

#### Constraint 9: Fix terminals and mandatory stations to stop
Force terminal and mandatory stations (set M) to be in the express set. 

In [17]:
# Force terminal stations to be in the express set
m.addConstr(z[0] == 1, name="terminal_start_in_express_set")
m.addConstr(z[STATION_COUNT - 1] == 1, name="terminal_end_in_express_set")

# Force mandatory stations to be in the express set
station_to_idx = {f"DT{i}": i - 1 for i in range(1, STATION_COUNT + 1)}

for station in M:
    m.addConstr(z[station_to_idx[station]] == 1, name=f"mandatory_express_{station}")

## Set objective and optimize

The objective minimizes the weighted total cost of the service plan:

`travel_time_cost + underservice_cost`

The first term is the passenger travel-time cost of stopping trains. If train `t` stops at station `j`, the model pays `k * q_j`, where `k` is the stop-time penalty from deceleration, dwell time, and acceleration, and `q_j` is the estimated number of onboard passengers affected by that stop.

The second term is the under-service penalty. If station `j` receives fewer stopping trains than its target `H_j`, the shortfall is `u_j`. The model pays `p_j * u_j`, where `p_j` is measured in passenger-minutes per missed stopping train and is higher for busier, more central, and interchange stations.

In [18]:
# Penalty of under-service p_j
# Logic:
# 1) use actual weekday 7-8am DTL station usage from transport_node_train_202603.csv
# 2) higher penalty near the CBD
# 3) higher penalty at interchange stations
# 4) one unit of u_j means one fewer stopping train than the target H_j

NUM_WEEKDAYS = 22 # for March 2026 specifically
CBD_STATIONS = {'DT16', 'DT17', 'DT18', 'DT19'}
INTERCHANGE_STATIONS = {'DT1', 'DT9', 'DT10', 'DT11', 'DT12', 'DT14', 'DT15', 'DT16', 'DT19', 'DT26', 'DT32', 'DT35'}

penalty_df = dtl_data[['PT_CODE', 'TOTAL_TAP_IN_VOLUME', 'TOTAL_TAP_OUT_VOLUME', 'volume']].copy()

# Average number of station users in the study hour on a weekday.
# e.g. if volume is 2200 for march weekdays at 7-8am, then 2200/22 = 100 users on average per weekday in that hour.
penalty_df['avg_hourly_station_users'] = penalty_df['volume'] / NUM_WEEKDAYS

# Map target service levels H_j; H[j] is aligned with dtl_stations[j].
# format: {'DT1': H[0], 'DT2': H[1], ...}
H_map = dict(zip(dtl_stations, H))
penalty_df['H'] = penalty_df['PT_CODE'].map(H_map)

# CBD proximity multiplier based on line-distance to downtown-core stations.
station_idx = {s: i for i, s in enumerate(dtl_stations)}
penalty_df['dist_to_cbd'] = penalty_df['PT_CODE'].map(
    lambda s: min(abs(station_idx[s] - station_idx[c]) for c in CBD_STATIONS if c in station_idx)
)
max_dist = penalty_df['dist_to_cbd'].max()
penalty_df['cbd_mult'] = 1 + 0.3 * (1 - penalty_df['dist_to_cbd'] / max_dist)
# if a station is far from CBD, then dist_to_cbd/max_dist is close to 1, so 1 + 0.3(1-1) = 1 so multiplier only 1.0
# if station is close to CBD, then dist_to_cbd/max_dist is close to 0, so 1 + 0.3(1-0) = 1.3 so multiplier up to 1.3

# Interchange bonus.
penalty_df['interchange_mult'] = np.where(
    penalty_df['PT_CODE'].isin(INTERCHANGE_STATIONS), 1.25, 1.0
)
# If station is an interchange, multiplier is 1.25, else 1.0

# Approximate users affected by one missed stop and the extra wait they incur.
penalty_df['users_per_missed_stop'] = penalty_df['avg_hourly_station_users']
penalty_df['extra_wait_per_missed_stop_min'] = 60 / penalty_df['H']
# e.g. if average hourly users is 240, target stops Hj = 24, then 240/24 = 10 users per missed stop
# assumes users are spread roughly evenly across the planned stopping trains

# Final under-service penalty in passenger-minutes per unit of u_j.
penalty_df['p_j'] = (
    penalty_df['users_per_missed_stop']
    * penalty_df['extra_wait_per_missed_stop_min']
    * penalty_df['cbd_mult']
    * penalty_df['interchange_mult']
).round(2)
# pj = (users affected by one missed stop) * (extra wait in minutes due to one missed stop) * (CBD multiplier) * (interchange multiplier)

# Force the same order as the optimization variables: p[j] must match dtl_stations[j].
penalty_df = penalty_df.set_index('PT_CODE').loc[dtl_stations].reset_index()
p = penalty_df['p_j'].tolist()

display(
    penalty_df[['PT_CODE', 'avg_hourly_station_users', 'H', 'cbd_mult', 'interchange_mult', 'p_j']]
)
print(p)

,PT_CODE,avg_hourly_station_users,H,cbd_mult,interchange_mult,p_j
0,DT1,9855.136364,24,1.01875,1.25,31374.75
1,DT2,1181.636364,18,1.03750,1.00,4086.49
2,DT3,1495.318182,18,1.05625,1.00,5264.77
3,DT4,519.727273,12,1.07500,1.00,2793.53
4,DT5,2641.500000,18,1.09375,1.00,9630.47
5,DT6,1486.318182,18,1.11250,1.00,5511.76
6,DT7,1080.454545,12,1.13125,1.00,6111.32
7,DT8,1991.045455,18,1.15000,1.00,7632.34
8,DT9,979.045455,24,1.16875,1.25,3575.81
9,DT10,1079.636364,24,1.18750,1.25,4006.46


[31374.75, 4086.49, 5264.77, 2793.53, 9630.47, 5511.76, 6111.32, 7632.34, 3575.81, 4006.46, 28726.4, 11718.24, 5573.7, 13588.37, 4680.38, 7643.96, 8688.53, 6444.26, 10648.92, 5740.0, 4924.04, 5444.99, 4591.8, 6595.17, 5609.86, 10603.09, 8174.41, 8265.71, 6638.93, 7531.79, 7241.75, 29399.01, 12592.3, 5592.94, 5407.67]


#### Check `q_j` Values
This display cell shows the passenger-impact weight `q_j` for each station. The values are already defined earlier so Constraint 6 can use them.

In [19]:
# q was already defined before Constraint 6.
# This cell only displays q[j] in station order for checking/reporting.
q_table = pd.DataFrame({
    'station': dtl_stations,
    'q_j': q,
})

print(f'STATION_COUNT: {len(q)}')

STATION_COUNT: 35


In [20]:
k = 1.45 
# k = 1.45 means that every stop costs each passengers 1.45 minutes
travel_time_cost = k * gp.quicksum(
    q[j] * x[t][j]
    for t in range(TRAIN_COUNT)
    for j in range(STATION_COUNT)
)

underservice_cost = gp.quicksum(
    p[j] * u[j]
    for j in range(STATION_COUNT)
)

# may exclude the costs
# express_cost = gp.quicksum(
#     F[t] * y[t]
#     for t in range(TRAIN_COUNT)
# )

m.setObjective(
    travel_time_cost + underservice_cost,
    GRB.MINIMIZE
)

m.optimize()


if m.status == GRB.OPTIMAL:
    express_trains = [t for t in range(TRAIN_COUNT) if y[t].X > 0.5]
    express_station_names = [dtl_stations[j] for j in range(STATION_COUNT) if z[j].X > 0.5]
    skipped_station_names = [dtl_stations[j] for j in range(STATION_COUNT) if z[j].X <= 0.5]

    print("Express train indices:", express_trains)
    print("Express route:", " -> ".join(express_station_names))
    print("Stations skipped by express trains:", skipped_station_names)
else:
    print("No optimal solution found. Status code:", m.status)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.2.0 25C56)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 3412 rows, 934 columns and 8907 nonzeros (Min)
Model fingerprint: 0x4b4e54b8
Model has 851 linear objective coefficients
Variable types: 35 continuous, 899 integer (899 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [7e+01, 3e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]

Found heuristic solution: objective 141842.51538
Presolve removed 2121 rows and 442 columns
Presolve time: 0.03s
Presolved: 1291 rows, 492 columns, 3786 nonzeros
Variable types: 0 continuous, 492 integer (474 binary)

Root relaxation: objective 1.391087e+05, 1017 iterations, 0.04 seconds (0.02 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time
